In [ ]:
import pandas as pd
import numpy as np
import xarray as xr
import pygmt

pygmt.config(FORMAT_FLOAT_MAP="%1.1e")  
reg = 'CT'
train_size = 1658
# Load the elevation from netcdf grid
grid_CT = xr.open_dataset('/mnt/beegfs/nragu/tsunami/ML4SicilyTsunami/data/processed/CT_defbathy.nc',engine='netcdf4')
#get lat long limits from the grid
ymin = grid_CT['y'].min().values
ymax = grid_CT['y'].max().values
xmin = grid_CT['x'].min().values
xmax = grid_CT['x'].max().values


dthresholds = np.array([
    1.000e-02, 5.000e-02, 1.000e-01, 2.000e-01,
    3.000e-01, 5.000e-01, 8.000e-01, 1.000e+00,
    1.500e+00, 2.000e+00, 2.500e+00, 3.000e+00,
    3.600e+00, 4.320e+00, 5.180e+00, 6.220e+00,
    7.460e+00, 8.950e+00, 1.074e+01, 1.289e+01,
    1.547e+01, 1.856e+01, 2.227e+01
]) * 100

def interpolate_ex_rate(ex_rate_table, thresholds, exact_depth):
    """
    Interpolate exceedance rates for a given exact depth in cm.
    If depth exists, returns that column. Else interpolates between two threshold columns.
    """
    # ✅ Ensure thresholds is a NumPy array
    thresholds = np.array(thresholds)

    print('thresholds:', thresholds)
    print('exact_depth:', exact_depth)

    # Case 1: Exact match
    if exact_depth in thresholds:
        col = np.where(thresholds == exact_depth)[0][0]
        print(f"Exact depth {exact_depth} cm found in column {col}. Returning ex_rate for that depth.")
        return ex_rate_table[:, col]

    # Case 2: Interpolation between nearest thresholds
    else:
        # Ensure the depth is within range
        if exact_depth < thresholds.min() or exact_depth > thresholds.max():
            raise ValueError(f"Depth {exact_depth} cm is outside threshold range ({thresholds.min()}–{thresholds.max()}).")

        lower_col = np.where(thresholds < exact_depth)[0][-1]
        upper_col = np.where(thresholds > exact_depth)[0][0]

        print(f"Exact depth {exact_depth} cm not found. Interpolating between {thresholds[lower_col]} and {thresholds[upper_col]} cm.")

        lower_val = ex_rate_table[:, lower_col]
        upper_val = ex_rate_table[:, upper_col]

        # Linear interpolation
        x0, x1 = thresholds[lower_col], thresholds[upper_col]
        interpolated_ex_rate = lower_val + (exact_depth - x0) * (upper_val - lower_val) / (x1 - x0)

        print('Non-zero sizes:', np.count_nonzero(lower_val), np.count_nonzero(upper_val))
        print('Final non-zero size:', np.count_nonzero(interpolated_ex_rate))

        return interpolated_ex_rate


#flattent the grid to 1D array to get x,y,z
grid_CT_flat = grid_CT.stack(points=['y', 'x'])
zero_mask = np.load('/mnt/beegfs/nragu/tsunami/ML4SicilyTsunami/data/processed/zero_mask_CT_892.npy')
zero_mask_flat = zero_mask.flatten()
np.count_nonzero(~zero_mask_flat)

#convert to pandas dataframe
grid_CT_flat = grid_CT_flat.to_dataframe()

fig = pygmt.Figure()

with fig.subplot(
        nrows=1,
        ncols=1,
        subsize = ["8","20c"],
        sharey = True,
        frame = ["ag"],
        clearance  = "w4.5c",
        ):
    with fig.set_panel(0):
        
        cptfile = '/mnt/beegfs/nragu/tsunami/ML4SicilyTsunami/scripts/PaperIIPlots/PaperI/r2/bathy.cpt'
        cmap = pygmt.makecpt(cmap=cptfile,continuous=False)
        fig.grdimage(grid_CT['z'], cmap=True, shading=True,region=[xmin,xmax,ymin,ymax],projection='M6c')
        fig.grdcontour(grid_CT['z'], levels=10, pen='0.5p,white', limit=[-100, 0],projection='M6c')
        cptfile = '/mnt/beegfs/nragu/tsunami/ML4SicilyTsunami/scripts/JGR/plots/extra/hot.cpt'
        cmap = pygmt.makecpt(cmap=cptfile,continuous=False)
        ex_rate_table = np.load(f'/mnt/beegfs/nragu/tsunami/ML4SicilyTsunami/data/simu/SIS/SIS_MAPS/catania/3000/hc_mean.npy')
        ex_rate = interpolate_ex_rate(ex_rate_table, dthresholds,500)
        ex_rate = np.reshape(ex_rate, (grid_CT.dims['y'], grid_CT.dims['x']))
        ex_rate = ex_rate.flatten()
        ex_rate = ex_rate[zero_mask_flat==False]
        idx_val = pd.read_csv(f'/mnt/beegfs/nragu/tsunami/ML4SicilyTsunami/model/{reg}/multifoldMC/PTHA/_Grid_scores_{train_size}_PSHalf.txt', sep=",")
        #drop rows with 0 ex_rate
        idx_val = idx_val.iloc[ex_rate > 0]
        ex_rate = ex_rate[ex_rate > 0]
        fig.plot(x=idx_val['lon'], y=idx_val['lat'],fill=ex_rate,style='s0.0075c',cmap = True,projection='M6c',frame = ["ag"])
        fig.colorbar(frame=["x+lprobability"], position="JML+o4.5c/-2.5c+w8c/0.5c+m", log = True)  
fig.show()


In [ ]:
import pandas as pd
import numpy as np
import xarray as xr
import pygmt

pygmt.config(FORMAT_FLOAT_MAP="%1.1e")  
#select particular representative gauge
reg = 'CT'
list_size = ['225', '529','892','1658','3454','7071']   
columnname = str(38)
mask_size = '892'
train_size = list_size[3]

# Load the elevation from netcdf grid
grid_CT = xr.open_dataset('../../../data/processed/CT_defbathy.nc',engine='netcdf4')
zero_mask = np.load('/mnt/beegfs/nragu/tsunami/ML4SicilyTsunami/data/processed/zero_mask_CT_892.npy')
zero_mask_flat = zero_mask.flatten()
np.count_nonzero(~zero_mask_flat)

#get lat long limits from the grid
ymin = grid_CT['y'].min().values
ymax = grid_CT['y'].max().values
xmin = grid_CT['x'].min().values
xmax = grid_CT['x'].max().values

Depth_list = [20,100,300,500] 

dthresholds = [1.000e-02, 5.000e-02, 1.000e-01, 2.000e-01,
       3.000e-01, 5.000e-01, 8.000e-01, 1.000e+00,
       1.500e+00, 2.000e+00, 2.500e+00, 3.000e+00,
       3.600e+00, 4.320e+00, 5.180e+00, 6.220e+00,
       7.460e+00, 8.950e+00, 1.074e+01, 1.289e+01,
       1.547e+01, 1.856e+01, 2.227e+01]

#convert m thresholds to cm
dthresholds = np.array(dthresholds) * 100  # convert to cm
ex_rate_table = np.load(f'/mnt/beegfs/nragu/tsunami/ML4SicilyTsunami/data/simu/SIS/SIS_MAPS/catania/3000/hc_mean.npy')

def interpolate_ex_rate(ex_rate_table, thresholds, exact_depth):
    """
    Interpolate exceedance rates for a given exact depth.
    If depth exists, returns that column. Else interpolates between two threshold columns.
    """
    if exact_depth in thresholds:
        col = np.where(thresholds == exact_depth)[0][0]
        print(f"Exact depth {exact_depth} cm found in column {col}. Returning ex_rate for that depth.")
        return ex_rate_table[:, col]
    else:
        lower_col = np.where(thresholds < exact_depth)[0][-1]
        upper_col = np.where(thresholds > exact_depth)[0][0]
        
        print(f"Exact depth {exact_depth} cm not found. Interpolating between {thresholds[lower_col]} and {thresholds[upper_col]} cm.")

        lower_val = ex_rate_table[:, lower_col]
        upper_val = ex_rate_table[:, upper_col]

        # linear interpolation: y = y0 + (x - x0)*(y1 - y0)/(x1 - x0)
        x0, x1 = thresholds[lower_col], thresholds[upper_col]
        print('non zero size',np.count_nonzero(lower_val), np.count_nonzero(upper_val))
        interpolated_ex_rate = lower_val + (exact_depth - x0) * (upper_val - lower_val) / (x1 - x0)
        print('final non zero size',np.count_nonzero(interpolated_ex_rate))

        return interpolated_ex_rate


for d, depth in enumerate(Depth_list):
    print(f"Processing depth: {depth} cm")

    
    # True, emulator, and SIS exceedance rates
    true_ref = np.load(
        f'/mnt/beegfs/nragu/tsunami/ML4SicilyTsunami/model/{reg}/multifoldMC/PTHA/true_PTHArate_53550.npy'
    )[:, d]

    emul_ref = np.load(
        f'/mnt/beegfs/nragu/tsunami/ML4SicilyTsunami/model/{reg}/multifoldMC/PTHA/pred_PTHArate_{train_size}.npy'
    )[:, d]

    sis_rate = interpolate_ex_rate(ex_rate_table, dthresholds,depth)
    sis_rate = np.reshape(sis_rate, (grid_CT.dims['y'], grid_CT.dims['x']))
    sis_rate = sis_rate.flatten()
    sis_ref = sis_rate[zero_mask_flat==False]

    # Errors
    def safe_log(x):
        #if x is v.small set to 0
        # x = np.where(x < 1e-8, 0, x)
        return np.where(x > 0, np.log10(x), 0)

    true_mask = true_ref > 0
    emul_mask = emul_ref > 0
    sis_mask = sis_ref > 0

    combined_mask_emul = true_mask & emul_mask
    combined_mask_sis = true_mask & sis_mask

    #when emul_ref is 0 or true_ref is 0, set error to negative of the other log value
    emul_error = safe_log(emul_ref) - safe_log(true_ref) 
    emul_error = np.where((emul_ref == 0) | (true_ref == 0), safe_log(true_ref) - safe_log(emul_ref), emul_error)
    
    sis_error = safe_log(sis_ref) - safe_log(true_ref)
    sis_error = np.where((sis_ref == 0) | (true_ref == 0), safe_log(true_ref) - safe_log(sis_ref), sis_error)

    # # --- PLOT LOG ERRORS ---
    fig = pygmt.Figure()

    with fig.subplot(
        nrows=2,
        ncols=1,
        subsize=["8", "17c"],
        sharey=True,
        frame=["ag"],
        clearance="w4.5c",
        margins=["0.1c", "0.1c"]
    ):

        # Panel 1: Emulator vs True
        with fig.set_panel(panel=[0, 0]):
            cptfile = './extra/bathy2.cpt'
            cmap = pygmt.makecpt(cmap=cptfile,continuous=False)
            fig.grdimage(grid_CT['z'], cmap=True, shading=True,region=[xmin,xmax,ymin,ymax],projection='M6c')
            cptfile = './extra/log_error.cpt'
            cmap = pygmt.makecpt(cmap=cptfile,continuous=False)
            idx_val = pd.read_csv(f'/mnt/beegfs/nragu/tsunami/ML4SicilyTsunami/model/{reg}/multifoldMC/PTHA/_Grid_scores_{train_size}_PSHalf.txt', sep=",")
            idx_val = idx_val.iloc[combined_mask_emul]
            emul_error = emul_error[combined_mask_emul]
            fig.plot(x=idx_val['lon'], y=idx_val['lat'],fill=emul_error,style='s0.0075c',cmap = True,projection='M6c',frame = ["ag"])
            # fig.colorbar(frame=["x+lerror"],position="JBC+o1c/4c+w8c/0.5c+h")
        # Panel 2: SIS vs True
        with fig.set_panel(panel=[1, 0]):
            cptfile = './extra/bathy2.cpt'
            cmap = pygmt.makecpt(cmap=cptfile,continuous=False)
            fig.grdimage(grid_CT['z'], cmap=True, shading=True,region=[xmin,xmax,ymin,ymax],projection='M6c')
            cptfile = './extra/log_error.cpt'
            cmap = pygmt.makecpt(cmap=cptfile,continuous=False)
            idx_val = pd.read_csv(f'/mnt/beegfs/nragu/tsunami/ML4SicilyTsunami/model/{reg}/multifoldMC/PTHA/_Grid_scores_{train_size}_PSHalf.txt', sep=",")
            idx_val = idx_val.iloc[combined_mask_sis]
            sis_error = sis_error[combined_mask_sis]
            fig.plot(x=idx_val['lon'], y=idx_val['lat'],fill=sis_error,style='s0.0075c',cmap = True,projection='M6c',frame = ["ag"])
    savefile = f'./PTHA_{train_size}_{reg}_Error_{depth}cm.png'
    # # fig.savefig(savefile, dpi=300)
    # # print(f"Saved {savefile}")
    fig.show()